In [ ]:
import os
import pandas as pd
import numpy as np
import openmatrix as omx
import geopandas as gpd

In [ ]:
input_data_folder = r'C:\projects\odot_joint_estimation\metro_data_prep\PopulationSim_to_RSG\HH_GQ_combined'
output_data_folder = r'C:\projects\odot_joint_estimation\pnr\SimOR\resident\model_data\metro\data_full'

assert os.path.exists(input_data_folder), f"Input data folder {input_data_folder} does not exist."
assert os.path.exists(output_data_folder), f"Output data folder {output_data_folder} does not exist."

maz_xwalk_new = pd.read_excel(os.path.join(input_data_folder, 'Metro_Block_MAZ_Xwalk-2025-12-16.xlsx'), sheet_name = "NEW - BlockWalk - 12-26-25")
maz_xwalk_old = pd.read_excel(os.path.join(input_data_folder, 'Metro_Block_MAZ_Xwalk-2025-12-16.xlsx'), sheet_name = "OLD - BlockWalk")

## Households Data Processing

In [ ]:
in_households = pd.read_csv(os.path.join(input_data_folder, 'households_all.csv'))

# Update MAZ/TAZ to new zone system
in_households = in_households.merge(
    maz_xwalk_old[['MAZ ID', 'FIPS (GEOID20)']],
    how='left',
    left_on='maz',
    right_on='MAZ ID'
)

in_households = in_households.merge(
    maz_xwalk_new[['FIPS (GEOID20)', 'MAZ NO', 'TAZ NO']],
    on = 'FIPS (GEOID20)',
    how = 'left'
)

hh_renaming_dict = {
    'hhid': 'household_id',
    # 'serialno': 'SERIALNO',  # (Optional, excluding for now)
    'MAZ NO': 'MAZ',
    'TAZ NO': 'TAZ',
    'htype': 'TYPE',
    'np': 'NP',
    'hht': 'HHT',
    'hhincadj': 'HHINCADJ',
    'nwrkrs_esr': 'WORKERS'
}

out_households = in_households[hh_renaming_dict.keys()].copy()
out_households.rename(columns=hh_renaming_dict, inplace=True)

out_households.head()

In [ ]:
# out_households.to_csv(os.path.join(output_data_folder, 'households.csv'), index=False)

## Persons Data Processing

In [ ]:
# Import Persons Table
in_persons = pd.read_csv(os.path.join(input_data_folder, 'persons_all.csv'))

# Update MAZ/TAZ to new zone system
in_persons = in_persons.merge(
    maz_xwalk_old[['MAZ ID', 'FIPS (GEOID20)']],
    how='left',
    left_on='maz',
    right_on='MAZ ID'
)

in_persons = in_persons.merge(
    maz_xwalk_new[['FIPS (GEOID20)', 'MAZ NO', 'TAZ NO']],
    on = 'FIPS (GEOID20)',
    how = 'left'
)

renaming_dict = {
    'hhid': 'household_id',
    'sporder': 'PERSON_NUM',
    'PERID': 'person_id',
    'PUMA': 'PUMA_GEOID',
    'TAZ NO': 'TAZ',  # optional, including for now
    'MAZ NO': 'MAZ',  # optional, including for now
    'agep': 'AGEP',
    'sex': 'SEX',
    'wkhp': 'WKHP',
    'wkw': 'WKW',
    'esr': 'ESR',
    'schg': 'SCHG',
    'mil': 'MIL',
    'occp': 'OCCCAT'
}

final_order = ['household_id','PERSON_NUM','person_id','PUMA_GEOID','AGEP','SEX','WKHP','WKW','ESR','SCHG','MIL','NAICSP','INDP',
'OCCP','OCCCAT','DDRS','DEAR','DEYE','DOUT','DPHY','DREM','TOLLFACTOR','FAREFACTOR']

# Grab data and rename columns
out_persons = in_persons[renaming_dict.keys()].copy()

out_persons.rename(columns=renaming_dict, inplace=True)

# fill in missing columns with empty strings
missing_cols = set(final_order) - set(out_persons.columns)
print(f"Filling missing columns in out_persons with empty string: {missing_cols}")

# Add new column headers to ActivitySim
out_persons[list(missing_cols)] = ''

# Reorder fields again
out_persons = out_persons[final_order]

out_persons.head()


In [ ]:
(~out_persons.household_id.isin(out_households.household_id)).sum()

## Land Use Processing

In [ ]:
landuse = pd.read_excel(os.path.join(input_data_folder, 'MAZ_23333_ActivitySim_Inputs_2025-11-20.xlsx'), sheet_name='ActivitySim_MAZ_inputs')

landuse_rename = {
    'MAZ_NO': 'MAZ',
    'TAZ_NO': 'TAZ',
}

# Rename columns
landuse = landuse.rename(columns=landuse_rename)

In [ ]:
# need to read in maz to stop file and merge with landuse
# maz_stop_walk = pd.read_csv(os.path.join(output_data_folder, 'maz_stop_walk.csv')).rename(columns={'maz': 'MAZ'})
# maz_stop_walk.head()

In [ ]:
# landuse = landuse.merge(maz_stop_walk, on='MAZ', how='left', validate='one_to_one')

#### Need to have landuse numbers match the skim numbers

In [ ]:
# skim_file = omx.open_file(os.path.join(output_data_folder, 'autoSkims__AM.omx'), 'r')
# skim_taz_numbers = range(1, skim_file.shape()[0] + 1)
# print(len(skim_taz_numbers))
# skim_file.close()


In [ ]:
# landuse.TAZ.nunique()

In [ ]:
# creating additional landuse rows for the missing TAZs
# missing_tazs = list(set(skim_taz_numbers) - set(landuse.TAZ))

# missing_landuse = pd.DataFrame(columns=landuse.columns, index=missing_tazs).fillna(0)
# missing_landuse['TAZ'] = missing_landuse.index
# missing_landuse['MAZ'] = landuse.MAZ.max() + range(len(missing_landuse))  # Giving dummy MAZ numbers for these TAZs
# missing_landuse['EXTERNAL'] = 1 # Assuming these missing TAZs are external, set EXTERNAL to 1
# landuse = pd.concat([landuse, missing_landuse], ignore_index=True)

In [ ]:
# FIXME additional columns needed for current SANDAG implementation
# expect these to be removed or changed in future versions
# landuse['micro_dist_local_bus'] = landuse['walk_dist_local_bus'] 
# landuse['micro_dist_premium_transit'] = landuse['walk_dist_premium_transit']
# landuse['MicroAccessTime'] = 0.5
# landuse['nev'] = 0
# landuse['microtransit'] = 0
# landuse['external_work'] = np.where(landuse['EXTERNAL'], 10, 0)  # adding non-zero size for external models so model won't crash
# landuse['external_nonwork'] = np.where(landuse['EXTERNAL'], 10, 0)  # adding non-zero size for external models so model won't crash
# landuse['external_MAZ'] = np.where(landuse['EXTERNAL'], 1, 0)
# landuse['totint'] = 10  # total intersections

In [ ]:
# assert landuse.TAZ.nunique() == len(skim_taz_numbers), "Landuse and skim file TAZ counts do not match."

In [ ]:
# len(skim_taz_numbers)

#### Park and Ride Data

In [ ]:
# pnr = pd.read_excel(os.path.join(input_data_folder, '../../2025_PnR_2162_TAZs.xlsx'), sheet_name=0)

# # number of official parking spaces provided by the transit agency in PNR zones in the MAZ
# # since the data was available only at the TAZ level, the parking spaces are all assigned to the first MAZ listed in land_use.csv within the TAZ
# landuse_unique_taz = landuse.drop_duplicates(subset=['TAZ'], keep='first')
# landuse_unique_taz = landuse_unique_taz.merge(pnr[['TAZ', 'CAPACITY']], on='TAZ', how='left')
# landuse = landuse.merge(landuse_unique_taz[['MAZ', 'CAPACITY']], on='MAZ', how='left')
# landuse['PNR_CAP_FORMAL'] = landuse['CAPACITY'].fillna(0).astype(int)
# landuse.drop(columns=['CAPACITY'], inplace=True)

# # number of informal parking spaces in PNR zones in the MAZ, e.g., surface street
# # sample a random 10 percent of zones with PNR lots
# landuse['PNR_CAP_INFORMAL'] = np.where(np.random.rand(len(landuse)) < 0.1, (landuse['PNR_CAP_FORMAL'] > 0) * np.random.randint(50, 200), 0)

# # cost of parking in the formal PNR lot (daily in 2024 dollars)
# # sample a random 50 percent of zones
# landuse['PNR_PRKCST'] = np.where(np.random.rand(len(landuse)) < 0.5, (landuse['PNR_CAP_FORMAL'] > 0) * np.random.randint(5, 20), 0)

# # average time in minutes to get from the formal parking lot to the transit stop
# # add dummy values to all PNR lots
# landuse['PNR_TERMINALTIME'] = (landuse['PNR_CAP_FORMAL'] > 0) * np.round(landuse['PNR_CAP_FORMAL'] / 50, decimals=0)

### Checking for consistency and saving to CSV

In [ ]:
# removing duplicated MAZ rows, keeping the first occurrence
landuse = landuse.drop_duplicates(subset=['MAZ'], keep='last')

In [ ]:
# checking to make sure all households have a valid MAZ in landuse
invalid_maz_hh = out_households[~out_households.MAZ.isin(landuse.MAZ)]
print(f"Removing {len(invalid_maz_hh)} households with invalid MAZs not found in landuse data.")
print(f"MAZs and TAZ of these households:\n{invalid_maz_hh[['MAZ', 'TAZ']].drop_duplicates().to_string(index=False)}")
out_households = out_households[out_households.MAZ.isin(landuse.MAZ)]


In [ ]:
# checking to make sure all persons have a valid household_id in households
invalid_hh_persons = out_persons[~out_persons.household_id.isin(out_households.household_id)]
print(f"Removing {len(invalid_hh_persons)} persons with invalid household_id not found in households data.")
out_persons = out_persons[out_persons.household_id.isin(out_households.household_id)]

In [ ]:
# assert landuse.TAZ.nunique() == len(skim_taz_numbers), "Landuse and skim file TAZ counts do not match."

In [ ]:
# Save to CSV
landuse.to_csv(os.path.join(output_data_folder, 'land_use.csv'), index=False)
out_persons.to_csv(os.path.join(output_data_folder, 'persons.csv'), index=False)
out_households.to_csv(os.path.join(output_data_folder, 'households.csv'), index=False)